# Problem description

**Context**
---
Churn prediction means detecting which customers are likely to cancel a subscription to a service based on how they use the service. It is a critical prediction for many businesses because acquiring new clients often costs more than retaining existing ones.

**Objective**
---

To build a model which would predict this user have exited or not

**Features**
---
  
 0.   RowNumber
 1.   CustomerId
 2.   Surname
 3.   CreditScore
 4.   Geography
 5.   Gender
 6.   Age
 7.   Tenure
 8.   Balance
 9.   NumOfProducts
 10.  HasCrCard
 11.  IsActiveMember
 12.  EstimatedSalary


**Label**
---

 1.  Exited  


# **Downloading dataset**

In [3]:
!pip install -q kaggle

In [4]:
from google.colab import files

In [ ]:
files.upload()

In [5]:
!mkdir ~/.kaggle

In [6]:
!cp kaggle.json ~/.kaggle/

In [7]:
!chmod 600 ~/.kaggle/kaggle.json

In [8]:
!mkdir dataset

In [9]:
%cd /content/dataset

/content/dataset


In [10]:
!kaggle datasets download -d "shrutimechlearn/churn-modelling"

  0% 0.00/262k [00:00<?, ?B/s]
100% 262k/262k [00:00<00:00, 57.6MB/s]


In [11]:
!unzip "/content/dataset/churn-modelling.zip"

Archive:  /content/dataset/churn-modelling.zip
  inflating: Churn_Modelling.csv     


# **Reading dataset**

In [12]:
import pandas as pd

In [13]:
df = pd.read_csv('/content/dataset/Churn_Modelling.csv')

# **Feature engineering**

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [15]:
df.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [16]:
# remove identifier features
df.drop(labels=['RowNumber',	'CustomerId',	'Surname'], axis=1, inplace=True)

In [17]:
df.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [18]:
# label encoding of Gender
# One hot encoding of Geography
df = pd.concat([df,
           pd.get_dummies(df['Gender'], prefix='Gender', drop_first=True),
           pd.get_dummies(df['Geography'], prefix='Geography')], axis=1)
df = df.drop(columns=['Gender', 'Geography'])

In [19]:
y = df['Exited']

In [20]:
X = df.drop(columns='Exited')

# **Splitting dataset**

In [21]:
from sklearn.model_selection import train_test_split

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y, random_state=42,)

# **Over sampling**

In [58]:
from imblearn.over_sampling import RandomOverSampler


In [59]:
oversample = RandomOverSampler()

In [60]:
X_train, y_train = oversample.fit_resample(X_train, y_train)

# **Feature Standardization**

In [61]:
from sklearn.preprocessing import StandardScaler

In [62]:
scaler = StandardScaler()

In [63]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Evaluating

In [64]:
from sklearn.metrics import classification_report

In [65]:
def compute_classification_report(y_true, y_pred):
  target_names = ['No', 'Yes']
  print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

# **KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
clf = KNeighborsClassifier(n_neighbors=13, p=2)

In [ ]:
clf.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=13)

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.8008    0.8382    0.8190      6086
         Yes     0.8454    0.8093    0.8269      6654

    accuracy                         0.8231     12740
   macro avg     0.8231    0.8237    0.8230     12740
weighted avg     0.8241    0.8231    0.8232     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.7602    0.9024    0.8252      1342
         Yes     0.6781    0.4195    0.5183       658

    accuracy                         0.7435      2000
   macro avg     0.7192    0.6609    0.6718      2000
weighted avg     0.7332    0.7435    0.7242      2000



# **Naive bayes**

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
clf = GaussianNB()

In [ ]:
clf.fit(X_train, y_train)

GaussianNB()

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.7433    0.6923    0.7169      6840
         Yes     0.6695    0.7229    0.6952      5900

    accuracy                         0.7064     12740
   macro avg     0.7064    0.7076    0.7060     12740
weighted avg     0.7092    0.7064    0.7068     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.7288    0.8931    0.8026      1300
         Yes     0.6585    0.3829    0.4842       700

    accuracy                         0.7145      2000
   macro avg     0.6936    0.6380    0.6434      2000
weighted avg     0.7042    0.7145    0.6912      2000



# **SVM**

In [ ]:
from sklearn.svm import SVC

In [ ]:
clf = SVC(C=1, kernel='rbf')

In [ ]:
clf.fit(X_train, y_train)

SVC(C=1)

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.8246    0.8166    0.8206      6433
         Yes     0.8148    0.8229    0.8188      6307

    accuracy                         0.8197     12740
   macro avg     0.8197    0.8197    0.8197     12740
weighted avg     0.8198    0.8197    0.8197     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.8098    0.9227    0.8626      1398
         Yes     0.7346    0.4967    0.5927       602

    accuracy                         0.7945      2000
   macro avg     0.7722    0.7097    0.7276      2000
weighted avg     0.7872    0.7945    0.7813      2000



# **DT**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
clf = DecisionTreeClassifier(criterion='gini',
                             splitter='best',
                             max_depth=None,
                             min_samples_split=2,
                             min_samples_leaf=1,
                             random_state=0)

In [ ]:
clf.fit(X_train, y_train)

DecisionTreeClassifier(random_state=0)

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     1.0000    1.0000    1.0000      6370
         Yes     1.0000    1.0000    1.0000      6370

    accuracy                         1.0000     12740
   macro avg     1.0000    1.0000    1.0000     12740
weighted avg     1.0000    1.0000    1.0000     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.8763    0.8649    0.8706      1614
         Yes     0.4644    0.4896    0.4767       386

    accuracy                         0.7925      2000
   macro avg     0.6704    0.6773    0.6736      2000
weighted avg     0.7968    0.7925    0.7946      2000



# **RF**

In [82]:
from sklearn.ensemble import RandomForestClassifier

In [164]:
clf = RandomForestClassifier(n_estimators=30, 
                             criterion='gini',
                             max_depth=None,
                             min_samples_split=8,
                             min_samples_leaf=2,
                             random_state=0)

In [165]:
clf.fit(X_train, y_train)

RandomForestClassifier(min_samples_leaf=2, min_samples_split=8, n_estimators=30,
                       random_state=0)

In [166]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [167]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.9776    0.9927    0.9851      6273
         Yes     0.9928    0.9779    0.9853      6467

    accuracy                         0.9852     12740
   macro avg     0.9852    0.9853    0.9852     12740
weighted avg     0.9853    0.9852    0.9852     12740



In [168]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.9083    0.8999    0.9041      1608
         Yes     0.6044    0.6276    0.6158       392

    accuracy                         0.8465      2000
   macro avg     0.7564    0.7637    0.7599      2000
weighted avg     0.8488    0.8465    0.8476      2000



# **AdaBoostClassifier**

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
clf = AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=0)

In [ ]:
clf.fit(X_train, y_train)

AdaBoostClassifier(random_state=0)

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.7984    0.7726    0.7853      6583
         Yes     0.7650    0.7915    0.7780      6157

    accuracy                         0.7817     12740
   macro avg     0.7817    0.7820    0.7817     12740
weighted avg     0.7823    0.7817    0.7818     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.7866    0.9213    0.8486      1360
         Yes     0.7371    0.4688    0.5731       640

    accuracy                         0.7765      2000
   macro avg     0.7618    0.6950    0.7108      2000
weighted avg     0.7707    0.7765    0.7604      2000



# **Gradient boosting**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
clf = GradientBoostingClassifier(
    learning_rate=0.1,
    n_estimators=100,
    max_depth=3,
    random_state=0
)

In [ ]:
clf.fit(X_train, y_train)

GradientBoostingClassifier(random_state=0)

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.8243    0.8086    0.8164      6494
         Yes     0.8049    0.8208    0.8128      6246

    accuracy                         0.8146     12740
   macro avg     0.8146    0.8147    0.8146     12740
weighted avg     0.8148    0.8146    0.8146     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.8129    0.9330    0.8688      1388
         Yes     0.7715    0.5131    0.6163       612

    accuracy                         0.8045      2000
   macro avg     0.7922    0.7230    0.7426      2000
weighted avg     0.8003    0.8045    0.7916      2000



# **XGBoost**

In [ ]:
import xgboost as xgb

In [ ]:
clf = xgb.XGBClassifier(learning_rate=0.1,
                        max_depth=3,
                        n_estimators=100, 
                        random_state=0)


In [ ]:
clf.fit(X_train, y_train)

XGBClassifier()

In [ ]:
y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

In [ ]:
compute_classification_report(y_train_pred, y_train)

              precision    recall  f1-score   support

          No     0.8188    0.7989    0.8087      6529
         Yes     0.7939    0.8142    0.8039      6211

    accuracy                         0.8064     12740
   macro avg     0.8064    0.8065    0.8063     12740
weighted avg     0.8067    0.8064    0.8064     12740



In [ ]:
compute_classification_report(y_test_pred, y_test)

              precision    recall  f1-score   support

          No     0.8048    0.9330    0.8642      1374
         Yes     0.7740    0.5032    0.6099       626

    accuracy                         0.7985      2000
   macro avg     0.7894    0.7181    0.7370      2000
weighted avg     0.7951    0.7985    0.7846      2000

